# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
from tqdm import tqdm

Note: you may need to restart the kernel to use updated packages.


### Data Preparation

In [26]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [27]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [28]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [29]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [30]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [31]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [32]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)
model = model.to("cpu")
with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [29]:
batch_size = 128
model = model.to(device)

val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=batch_size, shuffle=False, collate_fn=transformers.default_data_collator,num_workers=4
)
acc = 0
total = 0
for batch in val_loader:
    input_ids=batch["input_ids"].to(device)
    attention_mask=batch["attention_mask"].to(device)
    token_type_ids=batch["token_type_ids"].to(device)
    true_labels = batch["labels"].to(device)
    with torch.no_grad():
        predicted = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        predicted = torch.argmax(predicted.logits,dim=1)

    acc += torch.sum(true_labels == predicted).to("cpu").numpy()
    total += len(true_labels)

accuracy = acc/total

In [31]:
print("accuracy = ", accuracy)

accuracy =  0.9083848627256987


In [30]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [10]:
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2,
                                                                        id2label={0: "not_duplicate", 1: "duplicate"},
                                                                        label2id={"not_duplicate": 0, "duplicate": 1})

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [12]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [14]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

In [13]:
import evaluate
import numpy as np 

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    acc = accuracy_score(labels, predictions)    
    return {
        "accuracy": acc,
    }

In [21]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./deberta-v3-qqp",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=500,
    report_to=None,  # Disable wandb if not needed
    push_to_hub=False,  # Set to True if you want to push to Hugging Face Hub
)

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/usr/bin/ld: невозможно найти -laio: Нет такого файла или каталога
collect2: error: ld returned 1 exit status


MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)

In [ ]:
trainer.train()


In [ ]:
# Evaluate the model
print("\nEvaluating model...")
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

Реалзиация такая. Берем вопрос и пропускаем его вместе с полем text2 через токеназер и прогоняем это через весь набор данных. Затем подаём что получилось в модель, отбираем варианты с набольшей вероятностью дубликата и с вероятностью выше некотрого порога. Затем печатаем top-5 впоросов, или меньше если меньше прошло порог. Работает это всё не быстро.

In [ ]:
MAX_LENGTH = 128

def get_qestion_pairs_tkn(question):
    def get_qestion_pairs(examples):
        result = tokenizer(
            question,
            examples["text2"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
        )

        return result
    
    return get_qestion_pairs

In [101]:
import warnings
warnings.filterwarnings('ignore')

def find_similar(question,model,qqp,threshold = 0.8):
    qestion_pairs_tkn = get_qestion_pairs_tkn(question)
    qqp_preprocessed = qqp['validation'].map(qestion_pairs_tkn, batched=False)
    batch_size = 128
    model = model.to(device)

    val_loader = torch.utils.data.DataLoader(
        qqp_preprocessed, batch_size=batch_size, shuffle=False, collate_fn=transformers.default_data_collator,num_workers=1
    )
    pred_prob = np.array([])
    for batch in tqdm(val_loader):
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        token_type_ids=batch["token_type_ids"].to(device)
        with torch.no_grad():
            predicted = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            pred = torch.softmax(predicted.logits, dim=1).data.cpu().numpy()
            pred = pred[:,1]
            pred_prob = np.append(pred_prob,pred)
            
    indx = np.argsort(pred_prob)[::-1]
    threshold = 0.8 
    similar_questons = []
    for i in indx:
        if pred_prob[i]>threshold:
            similar_questons.append(qqp_preprocessed[i]['text2'])
        else:
            break
    return similar_questons

In [99]:
def print_similar_question(question):
    print(f"question is '{question}'")
    similar = find_similar(question,model,qqp,threshold = 0.8)
    if len(similar) == 0:
        print("no similar")
    else:
        print("similar questions:")
        for q in similar[:5]:
            print(q)

In [106]:
question = qqp['test'][0]['text1']
print_similar_question(question)

question is 'Would the idea of Trump and Putin in bed together scare you, given the geopolitical implications?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 316/316 [06:49<00:00,  1.30s/it]

similar questions:
AOSDHIADSOIHADSO DASODASHDASOH


In [102]:
question = qqp['test'][2]['text1']
print_similar_question(question)

question is 'Why don't people simply 'Google' instead of asking questions on Quora?'


  0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 316/316 [06:43<00:00,  1.28s/it]

similar questions:
Why do people bother to ask questions on Quora they could just google to get the answer?
Why do people bother to ask questions on Quora they could just google to get the answer?
Why do people bother to ask questions on Quora they could just google to get the answer?
Why do people bother to ask questions on Quora they could just google to get the answer?
Why do people bother to ask questions on Quora they could just google to get the answer?


In [103]:
question = qqp['test'][3]['text1']
print_similar_question(question)

question is 'Is it safe to invest in social trade biz?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 316/316 [06:51<00:00,  1.30s/it]

no similar


In [104]:
question = qqp['test'][4]['text1']
print_similar_question(question)

question is 'If the universe is expanding then does matter also expand?'


  0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 316/316 [06:49<00:00,  1.30s/it]

similar questions:
If vacuum energy is created as space expands, can infinite energy be created?
If vacuum energy is created as space expands, can infinite energy be created?
If vacuum energy is created as space expands, can infinite energy be created?
If vacuum gravitational and dark energy are created as universe expands without limit?
If vacuum gravitational and dark energy are created as universe expands without limit?


In [105]:
question = qqp['test'][1]['text1']
print_similar_question(question)

question is 'What are the top ten Consumer-to-Consumer E-commerce online?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 316/316 [06:43<00:00,  1.28s/it]

no similar


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>